In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score

# Load raw data fresh
df_raw = pd.read_csv('data/Telco-Customer-Churn.csv')
df_raw = df_raw.drop('customerID', axis=1)

# Fix TotalCharges
df_raw['TotalCharges'] = df_raw['TotalCharges'].str.strip().replace('', np.nan)
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges']).fillna(0)

# Split X/y
X = df_raw.drop('Churn', axis=1)
y = df_raw['Churn'].map({'No': 0, 'Yes': 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Group columns by type
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
onehot_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']

# Build the preprocessor — everything handled inside ColumnTransformer, no manual mapping needed
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('bin', OrdinalEncoder(), binary_cols),          # turns Yes/No, Male/Female into 0/1 automatically
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), onehot_cols),
    ('ord', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]), ['Contract'])
], remainder='passthrough')  # SeniorCitizen already 0/1, passes through as-is

telco_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42, class_weight='balanced'))
])

telco_pipeline.fit(X_train, y_train)
predictions = telco_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))

Accuracy: 0.752306600425834
Recall: 0.8123324396782842


In [2]:
import joblib
joblib.dump(telco_pipeline, 'telco_pipeline.pkl')

['telco_pipeline.pkl']